# STAR AI ULTRA — FAST CODER A100 (Qwen3-Coder 30B-A3B)

**الوضع الافتراضي: برمجة سريعة وإدارة STAR AI** مع نموذج Qwen3-Coder المتخصص في توليد الأكواد وقراءتها، وتحليل نتائج أدوات Linux.

- **الاختيار الأساسي:** `qwen3-coder:30b` (MoE بحوالي 30B إجمالي و3.3B معاملات نشطة؛ تنزيل quantized ~19GB).
- **بديل إذا لم تتوفر الموارد:** `qwen3.5:9b` أو `qwen3.5:4b`.
- يتم تشغيل النموذج على GPU السحابي من Google Colab، وليس على لابتوبك.
- من Colab اختر `Runtime → Change runtime type → GPU → A100 → High-RAM` إن توفر.
- نفّذ الخلايا بالترتيب؛ لا تحتاج مفتاح Kimi API لتشغيل النموذج المحلي.
- **لا تشغّل أي كود أو أوامر يقترحها النموذج دون مراجعة واعتماد.** دفتر Colab ينفّذ فقط تشخيصات محددة.
- Ollama يبقى على `127.0.0.1:11434` ولا توجد خدمة عامة أو ربط إنتاجي بخادم Railway.
- Colab جلسة مؤقتة مدفوعة بالوحدات، وقد تنتهي دون إنذار، ولا تضمن GPU أو الذاكرة في كل جلسة.
- لا تضع أي مفاتيح API أو بيانات سرية في الملفات العامة أو محادثات النموذج.

اختبار GitHub Actions يفحص صياغة الدفتر فقط؛ تشغيل النموذج على A100 يتطلب تنفيذ خلاياه في حساب Colab.

**المصدر:** https://ollama.com/library/qwen3-coder:30b

## 1) تحقق فعلي من GPU والذاكرة والمساحة
يُرفض التشغيل دون NVIDIA GPU، وتُقرأ الموارد من الجلسة الحالية بدلاً من افتراض سعة ثابتة.

In [ ]:
import os, sys, shutil, subprocess, json, time
from pathlib import Path

def run_cmd(args, timeout=30):
    return subprocess.run(args, capture_output=True, text=True, timeout=timeout, check=False)

gpu_check = run_cmd(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"])
if gpu_check.returncode != 0 or not gpu_check.stdout.strip():
    raise RuntimeError("No NVIDIA GPU detected. Select Runtime → Change runtime type → GPU.")

gpu_name, total_mib, free_mib = [
    v.strip() for v in gpu_check.stdout.splitlines()[0].rsplit(",", 2)
]
gpu_free_gib = int(free_mib) / 1024
disk_free_gib = shutil.disk_usage("/content").free / (1024 ** 3)
print("GPU:", gpu_name)
print("GPU total GiB:", round(int(total_mib) / 1024, 1))
print("GPU free GiB:", round(gpu_free_gib, 1))
print("Disk free GiB:", round(disk_free_gib, 1))
print("Python:", sys.version.split()[0])
print("Runtime location: Colab cloud; your laptop is not executing this model.")

## 2) اختيار أسرع نموذج متخصص للبرمجة (Fast Coder)

يختار الدفتر `qwen3-coder:30b` افتراضياً عندما تكفي ذاكرة GPU ومساحة القرص. هو نموذج MoE متخصص في البرمجة، ولا تعني 30B أنه يحتاج معالجة كل تلك المعاملات لكل token. الأداء الفعلي يحدده اختبار الاستدلال في Colab. التقديرات تشمل هامشاً للسياق وKV-cache، ولا تضمن عدم نفاد الذاكرة.

للاختيار اليدوي غيّر `MODEL_OVERRIDE` إلى اسم موجود في قائمة النماذج المؤهلة.

In [ ]:
MODEL_OVERRIDE = None   # Optional: "qwen3.5:9b" for a smaller model
# (minimum free GPU GiB, minimum free disk GiB) before downloads
MODEL_CHOICES = [
    ("qwen3-coder:30b", 30, 28), # coding-focused MoE, ~19GB download
    ("qwen3.5:9b", 12, 12),   # smaller, lower download cost
    ("qwen3.5:4b", 7, 8),
]
eligible = [name for name, need_gpu, need_disk in MODEL_CHOICES
            if gpu_free_gib >= need_gpu and disk_free_gib >= need_disk]
if not eligible:
    raise RuntimeError("Not enough free GPU memory or disk for configured models")
if MODEL_OVERRIDE is not None and MODEL_OVERRIDE not in eligible:
    raise ValueError(f"Unsupported override for measured resources. Eligible: {eligible}")
MODEL = MODEL_OVERRIDE or eligible[0]
CONTEXT = 8192  # raise after confirming successful performance
print("Fast Coding Model:", MODEL)
print("Available choices:", eligible)
print("Model download has NOT started yet.")

## 3) تثبيت Ollama من المصدر الرسمي
يحمّل السكربت الرسمي من `ollama.com`. افحص سياسة الثقة في مصادر التثبيت قبل تشغيله في أي بيئة مؤسسية.

In [ ]:
if shutil.which("ollama") is None:
    install_script = "/tmp/star-ai-ollama-install.sh"
    subprocess.run(["curl", "-fsSL", "https://ollama.com/install.sh",
                    "-o", install_script], check=True)
    subprocess.run(["bash", install_script], check=True)
if not shutil.which("ollama"):
    raise RuntimeError("Ollama installation failed")
print(run_cmd(["ollama", "--version"]).stdout.strip())

## 4) تشغيل الخادم داخل جلسة Colab فقط
لا تستخدم نفقاً عاماً أو رابطاً بدون مصادقة. تم ضبط مضيف Ollama على localhost حصراً.

In [ ]:
import requests

OLLAMA_HOST = "127.0.0.1:11434"
OLLAMA_URL = "http://" + OLLAMA_HOST
ollama_env = os.environ.copy()
ollama_env.update({
    "OLLAMA_HOST": OLLAMA_HOST,
    "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_MAX_LOADED_MODELS": "1",
    "OLLAMA_CONTEXT_LENGTH": str(CONTEXT),
    "OLLAMA_KEEP_ALIVE": "5m",
    "OLLAMA_FLASH_ATTENTION": "1",
    "OLLAMA_KV_CACHE_TYPE": "q8_0",
})

def ollama_ready():
    try:
        return requests.get(OLLAMA_URL + "/api/tags", timeout=2).ok
    except requests.RequestException:
        return False

if not ollama_ready():
    log_file = open("/tmp/star-ai-ultra-ollama.log", "a")
    ollama_process = subprocess.Popen(
        ["ollama", "serve"], env=ollama_env,
        stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True
    )
    for _ in range(60):
        if ollama_ready():
            break
        if ollama_process.poll() is not None:
            break
        time.sleep(1)
if not ollama_ready():
    tail = Path("/tmp/star-ai-ultra-ollama.log").read_text(errors="replace")[-2500:]
    raise RuntimeError("Ollama failed to start. Log tail:\n" + tail)
print("Local Ollama ready:", OLLAMA_URL)

## 5) تنزيل الأوزان (مرة لكل جلسة مؤقتة)
قد يستهلك نموذج 35B قرابة **22GB** من التنزيل. لا تُشغّل هذه الخلية إن كنت لا تريد استهلاك رصيد/حزمة إنترنت Colab. إذا لم تكفِ مساحة القرص، شغّل خلية الاختيار السابقة بعد ضبط `MODEL_OVERRIDE` إلى نموذج أصغر.

In [ ]:
print("Downloading model:", MODEL)
print("Available storage GiB before download:", round(shutil.disk_usage("/content").free / 2**30, 1))
subprocess.run(["ollama", "pull", MODEL], check=True, env=ollama_env)
print("Downloaded successfully:", MODEL)

## 6) اختبار محادثة حقيقي + إثبات هل التنفيذ على GPU
رسالة الرد وحدها لا تثبت استخدام GPU؛ يتم عرض `ollama ps` و`nvidia-smi` بعد تنفيذ الاستدلال.

In [ ]:
SYSTEM = (
    "You are STAR AI ULTRA Fast Coder, an Arabic-first software engineer and code reviewer "
    "for Python, TypeScript, Node.js and authorized defensive security tasks. Be exact; never claim to have run "
    "commands or accessed external systems unless a provided tool result proves it. "
    "For any impactful system change, require user approval."
)

def ask_model(messages, max_tokens=512):
    r = requests.post(
        OLLAMA_URL + "/api/chat",
        json={
            "model": MODEL, "messages": messages, "stream": False,
            "think": False,
            "options": {"num_ctx": CONTEXT, "num_predict": max_tokens,
                        "temperature": 0.35}, "keep_alive": "5m"
        },
        timeout=600
    )
    r.raise_for_status()
    payload = r.json()
    answer = (payload.get("message") or {}).get("content", "").strip()
    if not answer:
        raise RuntimeError("Model returned no text; inspect /tmp/star-ai-ultra-ollama.log")
    return answer, payload

reply, stats = ask_model([
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "اكتب سطرين بالعربية عن تحليل سجلات Linux الدفاعي."}
], max_tokens=180)
print("MODEL RESPONSE:\n", reply)
print("\nMODEL:", stats.get("model"))
print("Generated tokens:", stats.get("eval_count"))
print("\nGPU/CPU processor report (check PROCESSOR column):")
print(subprocess.run(["ollama", "ps"], env=ollama_env,
                     capture_output=True, text=True).stdout)
print("\nCurrent GPU usage:")
print(run_cmd(["nvidia-smi", "--query-gpu=memory.used,utilization.gpu",
               "--format=csv,noheader"]).stdout.strip())

## 7) محادثة تفاعلية مع STAR AI
اكتب `chat("سؤالك")` لتتحدث مع النموذج. حفظ الذاكرة هنا مؤقت داخل الجلسة وليس في GitHub أو Drive.

In [ ]:
history = [{"role": "system", "content": SYSTEM}]

def chat(question, max_tokens=1000):
    global history
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Question must not be empty")
    answer, _ = ask_model(
        history + [{"role": "user", "content": question.strip()}],
        max_tokens=max_tokens
    )
    history.extend([
        {"role": "user", "content": question.strip()},
        {"role": "assistant", "content": answer}
    ])
    history = [history[0]] + history[-10:]
    print("STAR AI ULTRA:\n" + answer)
    return answer

chat("صمم خدمة Python FastAPI لإدارة مهام STAR AI مع مصادقة JWT، تحقق صلاحيات، وسجل تدقيق، مع اختبارين pytest.", max_tokens=700)

## 7A) مراجعة ملفات STAR AI من مساحة العمل (قراءة فقط)
استنسخ مشروعك العام إلى `/content/star-ai-workspace` إذا احتجت تحليل الملفات. لا تضع الأسرار في المشروع العام، ولا تشغّل تلقائياً أوامر من المساعد. هذه الخلية لا تغيّر ملفات المشروع، وتحمي من قراءة المسارات خارج مساحة العمل.

In [ ]:
from pathlib import Path
WORKSPACE = Path("/content/star-ai-workspace").resolve()
PUBLIC_REPO = "https://github.com/abdulstarzeki001-sketch/linux-terminal.git"
CLONE_PROJECT = False  # Change to True only to clone your authorized public repo
if CLONE_PROJECT:
    if WORKSPACE.exists() and any(WORKSPACE.iterdir()):
        print("Workspace already exists; no changes made")
    else:
        subprocess.run(["git", "clone", "--depth", "1", PUBLIC_REPO, str(WORKSPACE)],
                       check=True, timeout=120)
        print("Public repo cloned into", WORKSPACE)
else:
    print("Repository clone disabled by default.")

def review_local_file(relative_file):
    import re
    path = (WORKSPACE / relative_file).resolve()
    if not WORKSPACE.exists() or not path.is_relative_to(WORKSPACE) or not path.is_file():
        raise ValueError("File must exist inside /content/star-ai-workspace")
    if path.is_symlink() or any(
        part.startswith(".env") or re.search(r"(secret|credential|token|private|id_rsa|pem|key)", part, re.I)
        for part in path.parts[len(WORKSPACE.parts):]
    ):
        raise ValueError("Potential secret or sensitive path is not allowed")
    if path.stat().st_size > 40000:
        raise ValueError("File too large; select a smaller file for review")
    data = path.read_text(encoding="utf-8", errors="replace")
    if len(data) > 25000:
        data = data[:25000] + "\\n[truncated]"
    prompt = ("Review the following code for correctness, maintainability and security. "
              "Return prioritized fixes and suggested diffs; do not claim to edit files. "
              "FILE: " + relative_file + "\\n" + data)
    answer, _ = ask_model([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": prompt}
    ], max_tokens=1400)
    print(answer)
    return answer

# Example after manually enabling clone:
# review_local_file("server/app.py")

## 8) تشخيص Linux الحقيقي — أوامر محددة وآمنة
هذه الأوامر تُنفّذ فعلياً على **خادم Colab**، وليس لابتوبك أو Railway. لا يُشغّل أي نص يأتي من النموذج كأمر shell.

In [ ]:
SAFE_DIAGNOSTICS = {
    "gpu": ["nvidia-smi"],
    "ram": ["free", "-h"],
    "disk": ["df", "-h", "/content"],
    "python": ["python", "--version"],
    "ollama": ["ollama", "ps"],
}

def diagnostic(name):
    if name not in SAFE_DIAGNOSTICS:
        raise ValueError("Allowed diagnostics: " + ", ".join(SAFE_DIAGNOSTICS))
    result = subprocess.run(SAFE_DIAGNOSTICS[name], shell=False,
                            capture_output=True, text=True, timeout=30,
                            env=ollama_env)
    print(result.stdout or result.stderr)
    return result.returncode

diagnostic("gpu")

## 9) حفظ محادثة غير حساسة اختيارياً في Google Drive
بشكل افتراضي **لا يتم تركيب Drive ولا حفظ محادثات**. إذا أردت الحفظ، غيّر `SAVE_TO_DRIVE=True`؛ سيطلب Colab تفويض حسابك. راجع المحتوى قبل الحفظ، ولا تحفظ بيانات عملاء أو مفاتيح API.

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    output_dir = Path("/content/drive/MyDrive/STAR_AI_ULTRA")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "chat_history.json"
    output_file.write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved:", output_file)
else:
    print("Drive persistence disabled. No chat files written.")

## 10) إيقاف التشغيل والحماية
- عند الانتهاء: Colab → **Runtime → Disconnect and delete runtime** لإنهاء استهلاك GPU.
- لا تستخدم جلسة Colab لاستضافة API إنتاجي دائم. للنشر المستمر استخدم خادم GPU مخصصاً، HTTPS، مصادقة، صلاحيات محددة، وسجلات تدقيق.
- `OLLAMA_HOST=127.0.0.1` يعني أن Ollama داخل الجلسة فقط، ولا يمكن استدعاؤه مباشرة من Railway أو Cloudflare.
- للتحقق من تحميل الأوزان على GPU راجع عمود `PROCESSOR` في `ollama ps`؛ إذا كان يستخدم CPU فالاختبار لا يحقق هدف التسريع.
- إن ظهرت أخطاء تحميل أو نفاد الذاكرة: أوقف النموذج ب`ollama stop MODEL` من خلية Python، اختر نموذجاً أصغر، وأعد تشغيل خلايا التنزيل والاختبار. مخرجات كل خطوة هي دليل النجاح الفعلي.
- لا تضع مفاتيح API أو روابط حساسة في Notebook عام على GitHub.

### مصادر
- [Qwen3.5 على Ollama](https://ollama.com/library/qwen3.5)
- [توثيق Ollama API](https://docs.ollama.com/api)
- [Google Colab FAQ](https://research.google.com/colaboratory/faq.html)